In [29]:
import sqlglot
from sqlglot import exp
import time
import json
import pandas as pd
from pathlib import Path

In [30]:
DIR_PATH = './asset/spider_2/'

In [31]:
#for there are 3 different databases in the spider_2 dataset, we will separate the statements into 3 lists
big_query_statements = []
sqlite_statements = []
snowflake_statements = []

In [32]:
for sql_file in Path(DIR_PATH + 'big_query/').glob('*.sql'):
    statement =  sql_file.read_text()
    big_query_statements.append(statement)

for sql_file in Path(DIR_PATH + 'sqlite/').glob('*.sql'):
    statement =  sql_file.read_text()
    sqlite_statements.append(statement)

for sql_file in Path(DIR_PATH + 'snowflake/').glob('*.sql'):
    statement =  sql_file.read_text()
    snowflake_statements.append(statement)

In [33]:
print('Big query statement: ' + big_query_statements[0])

print('SQLite statement: ' + sqlite_statements[0])

print('Snowflake statement: ' + snowflake_statements[0])

Big query statement: DECLARE start_date STRING DEFAULT '20170201';
DECLARE end_date STRING DEFAULT '20170228';

WITH visit AS (
    SELECT
        fullvisitorid,
        MIN(date) AS date_first_visit
    FROM
        `bigquery-public-data.google_analytics_sample.ga_sessions_*`
    WHERE
       _TABLE_SUFFIX BETWEEN start_date AND end_date
    GROUP BY fullvisitorid
),

transactions AS (
    SELECT
        fullvisitorid,
        MIN(date) AS date_transactions
    FROM
        `bigquery-public-data.google_analytics_sample.ga_sessions_*` AS ga,
        UNNEST(ga.hits) AS hits
    WHERE
        hits.transaction.transactionId IS NOT NULL
        AND
        _TABLE_SUFFIX BETWEEN start_date AND end_date
    GROUP BY fullvisitorid
),

device_transactions AS (
    SELECT DISTINCT
        fullvisitorid,
        date,
        device.deviceCategory
    FROM
        `bigquery-public-data.google_analytics_sample.ga_sessions_*` AS ga,
        UNNEST(ga.hits) AS hits
    WHERE
        hits.transactio

In [34]:
def extract_sql_metadata(sql: str, dialect: str) -> dict:
    """
    Hàm lõi phân tích Cây cú pháp trừu tượng (AST) của SQLGlot.
    """
    # Parse câu SQL theo từng dialect tương ứng
    parsed = sqlglot.parse_one(sql, read=dialect)
    
    # 1. Lấy danh sách bảng thật (Loại bỏ các bảng tạm/CTE trong mệnh đề WITH)
    cte_names = {cte.alias.lower() for cte in parsed.find_all(exp.CTE)}
    tables = set()
    for table in parsed.find_all(exp.Table):
        if table.name.lower() not in cte_names:
            tables.add(table.name.lower())
            
    # Hàm phụ trợ để lấy tên cột từ các node
    def get_columns(nodes):
        cols = set()
        for node in nodes:
            # exp.Column chỉ lấy tên cột gốc, tự động bỏ qua exp.Alias (như 'AS something')
            for col in node.find_all(exp.Column):
                cols.add(col.name.lower())
        return cols

    # 2. Bóc tách Cột theo từng Mệnh đề (Clause)
    select_cols = set()
    for select in parsed.find_all(exp.Select):
        for expr in select.expressions:
            select_cols.update(get_columns([expr]))

    where_cols = get_columns(parsed.find_all(exp.Where))
    group_cols = get_columns(parsed.find_all(exp.Group))
    order_cols = get_columns(parsed.find_all(exp.Order))
    
    # 3. Bóc tách Mệnh đề JOIN
    join_cols = set()
    join_types = set()
    for join in parsed.find_all(exp.Join):
        # Lấy loại JOIN (LEFT, INNER, OUTER...)
        side = str(join.args.get("side") or "").strip()
        kind = str(join.args.get("kind") or "").strip()
        j_type = f"{side} {kind}".strip().upper()
        join_types.add(j_type if j_type else "INNER")
        
        # Lấy cột trong điều kiện ON
        if join.args.get("on"):
            join_cols.update(get_columns([join.args["on"]]))
        # Lấy cột trong điều kiện USING
        if join.args.get("using"):
            for identifier in join.args["using"]:
                join_cols.add(identifier.name.lower())

    # 4. Trả về định dạng JSON dictionary như yêu cầu
    return {
        "tables": sorted(list(tables)),
        "columns_select": sorted(list(select_cols)),
        "join_columns": sorted(list(join_cols)),
        "columns_where": sorted(list(where_cols)),
        "columns_group": sorted(list(group_cols)),
        "columns_order": sorted(list(order_cols)),
        "join_types": sorted(list(join_types)),
        "timestamp": int(time.time())
    }

In [35]:
metadata_list = []

In [36]:
for statement in big_query_statements:
    metadata = extract_sql_metadata(statement, dialect='bigquery')
    
    metadata_list.append(metadata)

In [37]:
for statement in sqlite_statements:
    metadata = extract_sql_metadata(statement, dialect='sqlite')
    
    metadata_list.append(metadata)

In [38]:
for statement in snowflake_statements:
    metadata = extract_sql_metadata(statement, dialect='snowflake')
    
    metadata_list.append(metadata)

In [40]:
print(len(metadata_list))

256


In [41]:
with open ('./preprocessed-data/spider_2_metadata.json', 'w') as f:
    json.dump(metadata_list, f, indent=4)